**Import modules**

In [16]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import zipfile as zf

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print("Done importing modules....")

Done importing modules....


In [2]:
# create separate dataframes per purpose

# train csv
train_csv_path = '../data/train.csv'
train_df = pd.read_csv(train_csv_path)


# test csv
test_csv_path = '../data/test.csv'
test_df = pd.read_csv(test_csv_path)

print("Dataframes successfully created...")

Dataframes successfully created...


In [7]:
# configure for ml predictions
DECISION_THRESHOLD = 0.50

# inspect both df
print('TRAIN CSV METADATA:\n')
display(train_df.info())

print("=====" * 10)

print('TEST CSV METADATA:\n')
display(test_df.info())

print("=====" * 10)


TRAIN CSV METADATA:

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB


None

TEST CSV METADATA:

<class 'pandas.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            367 non-null    str    
 1   Gender             356 non-null    str    
 2   Married            367 non-null    str    
 3   Dependents         357 non-null    str    
 4   Education          367 non-null    str    
 5   Self_Employed      344 non-null    str    
 6   ApplicantIncome    367 non-null    int64  
 7   CoapplicantIncome  367 non-null    int64  
 8   LoanAmount         362 non-null    float64
 9   Loan_Amount_Term   361 non-null    float64
 10  Credit_History     338 non-null    float64
 11  Property_Area      367 non-null    str    
dtypes: float64(3), int64(2), str(7)
memory usage: 34.5 KB


None

**Header Normalization**

In [9]:
def main(df):
    
    df = df.copy()
    
    df = col_renamed(df)
    
    print('Renamed columns...\n')
    
    print(df.columns) 
    
    print("=====" * 10)
    
    return df


   
def col_renamed(col):
    return col.rename(columns=
    {
        'ApplicantIncome': 'Applicant_Income',
        
        'CoapplicantIncome': 'Coapplicant_Income',
        
        'LoanAmount': 'Loan_Amount'
    }
)


# process the training df
train_df = main(train_df)

# process the test df
test_df = main(test_df)    

Renamed columns...

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Applicant_Income', 'Coapplicant_Income',
       'Loan_Amount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area',
       'Loan_Status'],
      dtype='str')
Renamed columns...

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Applicant_Income', 'Coapplicant_Income',
       'Loan_Amount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='str')


**Feature Engineering**

This section transforms existing variables and creates new features to make the data more suitable for machine learning. This includes converting `Dependents` into a numerical format and creating `Total_Income` and `Loan_To_Income` features.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    
    df = df.copy()
    
    # dependents contain '3+' -> 3.0
    if "Dependents" in df.columns:
        df['Dependents'] = (
            df['Dependents'].astype(str)
            .str.replace(r'[^0-9.]', "", regex=True)
            .replace({"": np.nan})
            .astype(float)
        )
        
        
    # engineered features
    applicant_inc = df.get('Applicant_Income', pd.Series(0, index=df.index)).fillna(0)
    coapplicant_inc = df.get('Coapplicant_Income', pd.Series(0, index=df.index)).fillna(0)
    loan_amnt = df.get('Loan_Amount', pd.Series(np.nan, index=df.index))
    df['Total_Income'] = applicant_inc + coapplicant_inc
    denom = df['Total_Income'].replace({0:np.nan})
    df['Loan_To_Income'] = loan_amnt / denom

    return df

**Feature and Target Separation**

This section separates the target variable from the feature variables used for the model training. It first verifies that `Loan_Status` exists, converts the target variable from `Y/N` to a `1/0`, removes `Loan_Status` and `Loan_ID` form the feature set, applies feature engineering to `X`.

In [ ]:
# check target column
if "Loan_Status" not in train_df.columns:
  raise ValueError("Your train.csv must contain 'Loan_Status' (Y/N).")

# target variable
y = train_df['Loan_Status'].map({'Y':1, 'N':0}).astype(int)

# create df
X = train_df.drop(columns=['Loan_Status', 'Loan_ID'], errors='ignore').copy()

# engineer features
X = engineer_features(X)

**Train-Validation Split**

This section splits the data into training and validation sets. `test_size=0.2` reserves 20% of the data for validation, while the remaining 80% is used for training. `stratify=y` preserves the same proportion of target classes (`Loan_Status`) in both sets. `random_state=42` ensures that the same split is produced each time the code is run.

In [12]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

**Data Preprocessing**

This section prepares the numerical and categorical features for machine learning. Numerical features are handled using median imputation and standardization, while categorical features are handled using most-frequent imputation and one-hot encoding. A `ColumnTransformer` applies the appropriate preprocessing pipeline to each feature type and removes any columns not included in the defined feature lists.

In [13]:
# define the numerical features to be processed
numeric_features = [ c for c in [
    'Applicant_Income', 'Coapplicant_Income', 'Loan_Amount',
    'Loan_Amount_Term', 'Credit_History', 'Dependents',
    'Total_Income', 'Loan_To_Income'
] if c in X.columns ]


# define the categorical features to be processed
categorical_features = [ c for c in [
    'Gender', 'Married', 'Education',
    'Self_Employed', 'Property_Area'
] if c in X.columns ]


# preprocessing for numerical features
num_tf = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)


# preprocessing for categorical features
cat_tf = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')) # ignore categories that were not seen during training
    ]
)

# apply the appropriate preprocessing pipeline to each feature type
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_tf, numeric_features),
        ('cat', cat_tf, categorical_features),
    ],

    # remove any columns that were not included above
    remainder='drop'
)

**Model Training**

This section trains a Logistic Regression model using the preprocessed training data and evaluates its performance on the validation data. The model is combined with the preprocessing steps in a single pipeline. Performance is evaluated using accuracy, precision, recall, ROC-AUC, classification report, and a confusion matrix.


In [ ]:
# train logistic regression model

clf = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='liblinear',
    random_state=42
)

# pipeline that combines preprocessing and model training
pipe = Pipeline(
    [
        ('preprocess', preprocessor),
        ('clf', clf)
    ]
)

# train pipeline using training data
pipe.fit(X_train, y_train)

# predict class labels for the validation data
y_hat = pipe.predict(X_valid)

# predict probabilities for the positive class
y_prob = pipe.predict_proba(X_valid)[:, 1]

# display
print("SUPERVISED HOLD-OUT\n")

# percentage of correct predictions
print(f'Accuracy: {accuracy_score(y_valid, y_hat):.3f}')

# proportion of predicted positives that were actually positive
print(f'Precision: {precision_score(y_valid, y_hat, zero_division=0):.3f}')

# proportion of actual positives that were correctly identified
print(f'Recall: {recall_score(y_valid, y_hat, zero_division=0):.3f}')

# display the ROC-AUC using predicted probabilities
print(f'ROC-AUC: {roc_auc_score(y_valid, y_prob):.3f}\n')

# Display precision, recall, F1-score, and support for each class
print(classification_report(y_valid, y_hat, zero_division=0))

# confusion matrix showing actual vs predicted classes
cm = confusion_matrix(y_valid, y_hat, labels=[0,1])

# convert the confusion matrix into a labeled DataFrame
cm_df = pd.DataFrame(cm, index=['True N', 'True Y'], columns=['Pred N', 'Pred Y'])

# labeled confusion matrix
print(f"\nConfusion Matrix:\n {cm_df}")

SUPERVISED HOLD-OUT

Accuracy: 0.837
Precision: 0.892
Recall: 0.871
ROC-AUC: 0.872

              precision    recall  f1-score   support

           0       0.72      0.76      0.74        38
           1       0.89      0.87      0.88        85

    accuracy                           0.84       123
   macro avg       0.81      0.82      0.81       123
weighted avg       0.84      0.84      0.84       123


Confusion Matrix:
         Pred N  Pred Y
True N      29       9
True Y      11      74


**Interpretation:**

The Logistic Regression model achieved an accuracy of 83.7% on the hold-out set, correctly classifying 103 out of 123 applications. The model achieved a precision of 89.2% and recall of 87.1% for approved applications, indicating that it generally performed well in identifying approved applications while limiting incorrect approval predictions. The confusion matrix shows that the model correctly classified 74 approved and 29 rejected applications, while incorrectly classifying 9 rejected applications as approved and 11 approved applications as rejected. The model achieved a ROC-AUC of 0.872, indicating good ability to distinguish between approved and rejected applications.



**Cross-Validation and Hyperparameter Tuning**

This section evaluates the Logistic Regression pipeline using stratified 5-fold cross-validation and performs light hyperparameter tuning on the `C` parameter. The best-performing pipeline is then refitted on all labeled data and saved using Joblib.

In [17]:
# stratified 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# evaluate ROC-AUC using cross-validation
cv_auc = cross_val_score(
    pipe,
    X,
    y,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

# evaluate accuracy using cross-validation
cv_acc = cross_val_score(
    pipe,
    X,
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)


# display mean and standard deviation of CV scores
print(f'\nCV AUC: mean={cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'\nCV ACC: mean={cv_acc.mean():.3f} ± {cv_acc.std():.3f}')

# test different C values and select the best one
param_grid = {'clf__C': [0.1, 1.0, 5.0, 10.0]}

# create grid search using ROC-AUC as the selection metric
grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)

# find the best C value using cross-validation
grid.fit(X, y)

# get the pipeline with the best parameters
best_pipe = grid.best_estimator_

# display the best parameter found
print(f"Best params: {grid.best_params_}")

# refit the best pipeline using all labeled data and save it
best_pipe.fit(X, y)
joblib.dump(best_pipe, 'loan_approval_model.joblib')
print("Saved -> loan_approval_model.joblib")

# check approval predictions using the selected decision threshold
train_probs = best_pipe.predict_proba(X)[:, 1]

# convert probabilities into class predictions
train_preds = (train_probs >= DECISION_THRESHOLD).astype(int)

# calculate the proportion of denied and approved predictions
share = pd.Series(train_preds).value_counts(normalize=True).rename({0:'Denied', 1: 'Approved'})


# display the decision threshold and approval share
print(f"\nDecision Threshold = {DECISION_THRESHOLD:.2f}")
print("Approval share on training set: ")
print(share)


CV AUC: mean=0.755 ± 0.036

CV ACC: mean=0.766 ± 0.034
Best params: {'clf__C': 0.1}
Saved -> loan_approval_model.joblib

Decision Threshold = 0.50
Approval share on training set: 
Approved    0.765472
Denied      0.234528
Name: proportion, dtype: float64


**Interpretation:**

The logistic regression model achieved a mean cross-validation ROC-AUC of 0.755 and accuracy of 76.6%, with relatively small variation across the validation folds. Hyperparameter tuning identified C=0.1 as the best parameter. Using a decision threshold of 0.50, the model classified approximately 76.5% of the training applications as approved and 23.5% as denied.

**Generate Test Predictions**

This section applies the final trained pipeline to the unseen test data. The test data is prepared using the same feature engineering process used during training, after which the model generates loan approval probabilities and Y/N predictions. The results, along with the corresponding `Loan_ID`, are saved to `loan_predictions.csv`.

In [18]:
# keep the test set IDs for the prediction output
test_ids = test_df.get('Loan_ID')

# remove the ID and apply the same feature engineering used during training
Xt = engineer_features(
    test_df.drop(columns=['Loan_ID'], errors='ignore').copy()
)

# predict the probability of loan approval
probs = best_pipe.predict_proba(Xt)[:, 1]

# convert probabilities into Y/N predictions using the decision threshold
preds = np.where(probs >= DECISION_THRESHOLD, "Y", "N")

# create a DataFrame containing predictions and approval probabilities
out = pd.DataFrame({
    'Loan_Status_Pred': preds,
    'Approval_Probability': probs
})

# add Loan_ID back to the output if it exists
if test_ids is not None:
    out.insert(0, "Loan_ID", test_ids.values)

# save the predictions to a CSV file
out.to_csv("loan_predictions.csv", index=False)

# confirm that the prediction file was created
print('Wrote predictions -> loan_predictions.csv')

Wrote predictions -> loan_predictions.csv


**Generate Output**

This section loads the generated prediction CSV into a pandas DataFrame for inspection and further analysis.

In [ ]:
# define the path to the prediction output file
output_csv = r'loan_predictions.csv'

# load the prediction csv into a dataframe
output = pd.read_csv(output_csv)

In [21]:
# original training data
display(train_df.head())

# features used for model training
display(X.head())

# test data before prediction
display(test_df.head())

# final model predictions
display(output.head())

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Property_Area,Total_Income,Loan_To_Income
0,Male,No,0.0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,5849.0,NaN
1,Male,Yes,1.0,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,6091.0,0.021015
2,Male,Yes,0.0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,3000.0,0.022000
3,Male,Yes,0.0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,4941.0,0.024287
4,Male,No,0.0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,6000.0,0.023500


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Property_Area
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban


,Loan_ID,Loan_Status_Pred,Approval_Probability
0,LP001015,Y,0.703398
1,LP001022,Y,0.647817
2,LP001031,Y,0.619550
3,LP001035,Y,0.691840
4,LP001051,N,0.496777


**Conclusion**

The machine learning approach successfully developed a Logistic Regression model to predict bank loan approval outcomes using applicant demographic, financial, and loan-related features. Feature engineering was used to create additional variables such as `Total_Income` and `Loan_To_Income`, while preprocessing handled missing values, numerical scaling, and categorical encoding within a unified pipeline.

On the hold-out validation set, the model achieved an accuracy of **83.7%**, precision of **89.2%**, recall of **87.1%**, and ROC-AUC of **87.2%**, indicating that the model was able to distinguish between approved and denied applications reasonably well. However, 5-fold cross-validation produced a lower average ROC-AUC of **75.5% ± 3.6%** and accuracy of **76.6% ± 3.4%**, suggesting that performance varies depending on the data split and that the single hold-out result may be somewhat optimistic.

Light hyperparameter tuning selected **C = 0.1** as the best value based on ROC-AUC. The resulting pipeline was refitted using the available labeled data and saved with Joblib for reuse. The final model was then applied to the unseen test dataset to generate both loan approval predictions and approval probabilities.

Overall, this project demonstrates that Logistic Regression can provide a useful baseline for loan approval prediction. The results also provide a foundation for future improvements, such as comparing other classification algorithms, performing more extensive hyperparameter tuning, evaluating different decision thresholds, and further investigating which applicant characteristics contribute most to loan approval predictions.
